# Setup

In [ ]:
%matplotlib inline

import os, sys, warnings, math

import numpy as np
import pandas as pd
from tqdm import tqdm
import src.utils as utils
from src.neural_network import ModelDataManager

import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 8})
plt.rcParams["svg.fonttype"] = "none"
plt.rc('font', family='Arial')
import seaborn as sns
sns.set_style("white")

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

# Files and parameters

In [ ]:
# directories and files
model_params_name = 'model_params_202602'
username = os.getenv('USER')
if sys.platform == 'darwin':
    if username == 'ahmad':
        datadir = '/Users/ahmad/software/snaplab_github/neuro_rnn/data'
        # modeldir = '/Users/ahmad/data/rutgers/neuro_rnn/results/pytorch/model/' + model_params_name
        modeldir = '/Volumes/Sabrent_2TB/rutgers/neuro_rnn/data/' + model_params_name
        outdir = modeldir
elif sys.platform == 'linux':
    if username == 'ab2792':
        datadir = '/home/ab2792/software/snaplab_github/neuro_rnn/data'
        modeldir = '/home/ab2792/data/neuro_rnn/results/pytorch/model'
        outdir = modeldir
    elif username == 'lindenmp':
        datadir = '/home/lindenmp/research_projects/neuro_rnn/data'
        modeldir = '/media/lindenmp/storage_ssd/research_projects/neuro_rnn/results/model_cpu'
        outdir = '/home/lindenmp/research_projects/neuro_rnn/results/figs'

model_params_file = os.path.join(datadir, model_params_name + '.csv')

check_only = False
verbose = True
save_figs = False

# select rows from model params CSV
rows_to_plot = []
# rows_to_plot = list(np.arange(54,57))

model_params, task_names, n_tasks, kernel_labels, n_kernels = utils.get_params_dataframe(
    model_params_file, rows=rows_to_plot, verbose=verbose)
n_models = len(model_params)

display(model_params)

# Load data

In [ ]:
if check_only:
    
    found_count = 0
    for model_idx in range(n_models):
        this = model_params.iloc[model_idx]
        file_path = os.path.join(modeldir, this.file_str_outputs)
        file_exists = os.path.isfile(file_path)
        if file_exists:
            found_count += 1
        else:
            print(f'Not found ... {this.file_str_outputs} ({model_idx})')
    print(f'Found {found_count}/{n_models} files.')
    
else:
    
    loaded_accuracy = []
    loaded_loss = []
    loaded_loss_task = []
    loaded_loss_spatial = []
    
    for model_idx in tqdm(np.arange(n_models), desc='Loading data'):
        
        # get model details
        this = model_params.iloc[model_idx]
        utils.check_if_supported(task=this.task_no_modifier, modifier=this.task_modifier)

        # get data file name
        file_path = os.path.join(modeldir, this.file_str_outputs)

        try:
            output_manager = ModelDataManager(file_path)
            
            # load test accuracy
            loaded_accuracy.append(output_manager.load_key_across_runs('test_accuracy'))
            
            # load training losses
            try:
                loaded_loss.append(output_manager.load_key_across_runs('training_loss'))
                loaded_loss_task.append(output_manager.load_key_across_runs('training_loss_task'))
                loaded_loss_spatial.append(output_manager.load_key_across_runs('training_loss_spatial'))
            except Exception as e:
                print(f"Loss data not available for model {model_idx}: {e}")
                loaded_loss.append(None)
                loaded_loss_task.append(None)
                loaded_loss_spatial.append(None)
                
        except:
            print("An error occurred:", sys.exc_info()[0])
            print("Exception message:", sys.exc_info()[1])

# Test accuracy

In [ ]:
if not check_only:
    
    color_palette = utils.get_my_colors(cat_trio=False, as_list=True)

    # initialise figures
    size_scale = 2.0
    font_size = int(10 * size_scale)
    n_fig_columns = min((3, n_tasks))
    n_fig_rows = math.ceil(n_tasks / n_fig_columns)
    fig_size_w = 3.5 * n_fig_columns * size_scale
    fig_size_h = 3.5 * n_fig_rows * size_scale
    f_plots, ax_plots = plt.subplots(n_fig_rows, n_fig_columns, figsize=(fig_size_w, fig_size_h),
                                     squeeze=True, sharex=False, sharey=True)
    f_violin, ax_violin = plt.subplots(n_fig_rows, n_fig_columns, figsize=(fig_size_w, fig_size_h),
                                       squeeze=True, sharex=False, sharey=True)
    plt.rcParams['font.size'] = font_size
    xlims_plots = np.zeros(n_tasks)
    violin_bodies = []
    all_means = [{} for _ in range(n_tasks)]

    for model_idx in tqdm(np.arange(n_models), desc='Plotting'):
        
        this = model_params.iloc[model_idx]
        
        try:
            test_accuracy_all_runs = loaded_accuracy[model_idx]
            
            # extract accuracy from each run
            test_accuracy = np.zeros((this.n_runs, test_accuracy_all_runs[0].shape[0] - 1))
            for run in np.arange(this.n_runs):
                test_accuracy[run, :] = test_accuracy_all_runs[run][1:] * 100

            n_epochs_actual = test_accuracy.shape[1] * 100
            n_logged_epochs = int(n_epochs_actual / 100)
            if n_logged_epochs > 5000:
                n_logged_epochs = 125
                n_epochs_actual = n_logged_epochs * 100
                test_accuracy = test_accuracy[:, :n_logged_epochs]
            x_step = int(n_epochs_actual / n_logged_epochs)
            x = np.arange(x_step, n_epochs_actual + x_step, x_step)

            # compute mean and CI across runs
            accuracy_mean_runs = test_accuracy.mean(axis=0)
            accuracy_std_runs = test_accuracy.std(axis=0)
            ci = 1.96 * (accuracy_std_runs / np.sqrt(test_accuracy.shape[0]))
            ci_lower = accuracy_mean_runs - ci
            ci_upper = accuracy_mean_runs + ci
            
            # compute mean of each run across time
            accuracy_mean_epochs = test_accuracy.mean(axis=1)
            
            all_means[this.task_index][this.kernel_index] = accuracy_mean_runs

            # plot mean accuracy vs. time
            f_plots.axes[this.task_index].plot(x, accuracy_mean_runs,
                                              color=color_palette[this.kernel_index],
                                              label=str(this.kernel_label))
            f_plots.axes[this.task_index].fill_between(x, ci_lower, ci_upper,
                                                       color=color_palette[this.kernel_index], alpha=0.1)
            xlims_plots[this.task_index] = np.max((xlims_plots[this.task_index], n_epochs_actual))
            
            # plot time-averaged accuracy as violin plot
            violin_parts = f_violin.axes[this.task_index].violinplot(
                dataset=accuracy_mean_epochs, positions=[this.kernel_index], showmedians=True)
            for partname in ('cbars', 'cmins', 'cmaxes', 'cmedians'):
                vp = violin_parts[partname]
                vp.set_edgecolor(color_palette[this.kernel_index])
            for pc in violin_parts['bodies']:
                pc.set_facecolor(color_palette[this.kernel_index])
                pc.set_edgecolor(color_palette[this.kernel_index])
            if this.task_index == 0:
                violin_bodies.append(violin_parts['bodies'][0])
            
        except:
            print("An error occurred:", sys.exc_info()[0])
            print("Exception message:", sys.exc_info()[1])

    # set title and labels for line plots
    for task_index in np.arange(n_tasks):
        xmax = xlims_plots[task_index]
        f_plots.axes[task_index].set_xlim([0, xmax + 10])
        f_plots.axes[task_index].set_ylim([0, 105])
        f_plots.axes[task_index].set_xticks(np.arange(0, xmax + 10, int(xmax / 4)))
        f_plots.axes[task_index].set_yticks(np.arange(0, 100 + 1, 20))
        f_plots.axes[task_index].tick_params(bottom=True, left=True)
        if task_index == 0:
            f_plots.axes[task_index].legend(loc='lower right')
        f_plots.axes[task_index].set_title(utils.get_task_label(task_names[task_index]),
                                           fontsize=font_size)
        f_plots.axes[task_index].grid()
    f_plots.text(0.51, 0.0, 'Epoch', ha='center', va='center')
    f_plots.text(0.0, 0.5, 'Test Accuracy (%)', ha='center', va='center', rotation='vertical')
    
    # set title and labels for violin plots
    for task_index in np.arange(n_tasks):
        f_violin.axes[task_index].set_ylim([0, 105])
        f_violin.axes[task_index].set_xticks((-1,))
        f_violin.axes[task_index].set_xticklabels([' '])
        f_violin.axes[task_index].set_yticks(np.arange(0, 100 + 1, 20))
        f_violin.axes[task_index].tick_params(bottom=True, left=True)
        if task_index == 0:
            f_violin.axes[task_index].legend(violin_bodies, kernel_labels, loc='lower right')
        f_violin.axes[task_index].set_title(utils.get_task_label(task_names[task_index]),
                                            fontsize=font_size)
        f_violin.axes[task_index].grid()
    f_violin.text(0.0, 0.5, 'Learning Speed (a.u.)', ha='center', va='center', rotation='vertical')
    
    sns.despine(fig=f_plots, offset=0, trim=False, left=False, right=True, top=True, bottom=False)
    sns.despine(fig=f_violin, offset=0, trim=False, left=False, right=True, top=True, bottom=False)
    
    f_plots.tight_layout()
    f_violin.tight_layout()
    plt.show()
    
    # save svg
    if save_figs:
        models_file = os.path.splitext(os.path.basename(model_params_file))[0]
        f_plots.savefig(os.path.join(outdir, 'accuracy_{:}_plots.svg'.format(models_file)),
                        dpi=300, bbox_inches='tight', pad_inches=0.01)
        f_violin.savefig(os.path.join(outdir, 'accuracy_{:}_violin.svg'.format(models_file)),
                         dpi=300, bbox_inches='tight', pad_inches=0.01)

# Training loss

In [ ]:
if not check_only:
    
    color_palette = utils.get_my_colors(cat_trio=False, as_list=True)
    loss_labels = ['Total Loss', 'Task Loss', 'Spatial Loss']
    loss_data_list = [loaded_loss, loaded_loss_task, loaded_loss_spatial]

    size_scale = 2.0
    font_size = int(10 * size_scale)
    plt.rcParams['font.size'] = font_size

    # create one figure per loss type
    for loss_type_idx, (loss_data, loss_label) in enumerate(zip(loss_data_list, loss_labels)):
        
        n_fig_columns = min((3, n_tasks))
        n_fig_rows = math.ceil(n_tasks / n_fig_columns)
        fig_size_w = 3.5 * n_fig_columns * size_scale
        fig_size_h = 3.5 * n_fig_rows * size_scale
        f_loss, ax_loss = plt.subplots(n_fig_rows, n_fig_columns, figsize=(fig_size_w, fig_size_h),
                                       squeeze=True, sharex=False, sharey=True)
        xlims = np.zeros(n_tasks)
        
        for model_idx in range(n_models):
            
            this = model_params.iloc[model_idx]
            
            if loss_data[model_idx] is None:
                print(f"No {loss_label.lower()} data for model {model_idx}")
                continue
            
            try:
                loss_all_runs = loss_data[model_idx]
                n_runs = min(this.n_runs, len(loss_all_runs))
                
                # extract loss from each run
                min_len = min(len(loss_all_runs[r]) for r in range(n_runs))
                loss_array = np.zeros((n_runs, min_len))
                for run in range(n_runs):
                    loss_array[run, :] = loss_all_runs[run][:min_len]
                
                # create x axis (epochs)
                x = np.arange(1, min_len + 1)
                
                # compute mean and CI across runs
                loss_mean = loss_array.mean(axis=0)
                loss_std = loss_array.std(axis=0)
                ci = 1.96 * (loss_std / np.sqrt(n_runs))
                ci_lower = loss_mean - ci
                ci_upper = loss_mean + ci
                
                # plot
                ax = f_loss.axes[this.task_index] if hasattr(f_loss, 'axes') else f_loss
                ax.plot(x, loss_mean, color=color_palette[this.kernel_index],
                        label=str(this.kernel_label))
                ax.fill_between(x, ci_lower, ci_upper,
                               color=color_palette[this.kernel_index], alpha=0.1)
                xlims[this.task_index] = max(xlims[this.task_index], min_len)
                
            except Exception as e:
                print(f"Error plotting {loss_label} for model {model_idx}: {e}")
        
        # polish plots
        axes_list = f_loss.axes if hasattr(f_loss, 'axes') else [f_loss]
        for task_index in range(n_tasks):
            ax = axes_list[task_index]
            xmax = xlims[task_index]
            if xmax > 0:
                ax.set_xlim([0, xmax])
            ax.tick_params(bottom=True, left=True)
            if task_index == 0:
                ax.legend(loc='upper right')
            ax.set_title(utils.get_task_label(task_names[task_index]), fontsize=font_size)
            ax.grid()
        f_loss.text(0.51, 0.0, 'Epoch', ha='center', va='center')
        f_loss.text(0.0, 0.5, loss_label, ha='center', va='center', rotation='vertical')
        
        sns.despine(fig=f_loss, offset=0, trim=False, left=False, right=True, top=True, bottom=False)
        f_loss.tight_layout()
        plt.show()
        
        # save svg
        if save_figs:
            models_file = os.path.splitext(os.path.basename(model_params_file))[0]
            loss_suffix = loss_label.lower().replace(' ', '_')
            f_loss.savefig(os.path.join(outdir, '{:}_{:}.svg'.format(loss_suffix, models_file)),
                           dpi=300, bbox_inches='tight', pad_inches=0.01)